# Notebook Analyse Five-Point Task (MRCP / Potentiels moteurs)

Ce carnet prolonge le pipeline présenté dans `01_preprocessing_notebook.ipynb` pour explorer les potentiels moteurs (Movement-Related Cortical Potentials, MRCP) du paradigme Five-Point. Comme dans les notebooks Go/NoGo et Textsemantic :
- **Type 1 — prêt à exécuter** : blocs autonomes que vous pouvez lancer tels quels.
- **Type 2 — à personnaliser** : cellules contenant des paramètres à ajuster (balises `A_COMPLETER`, exemples à modifier).
- **Type 3 — exploration** : idées de prolongements pour approfondir l'analyse (nouveaux canaux, métriques, comparaisons).

Le Five-Point test consiste à déplacer un curseur vers cinq cibles successives avec un crayon optique. Les MRCP apparaissent avant le mouvement (composante négative lente) et reflètent la préparation motrice. Nous allons :
1. Charger un fichier prétraité (`*_processed.fif`) pour un sujet donné.
2. Remapper les annotations vers les évènements clés (`onset`, `first_touch`, `retouch`) et construire des epochs alignées sur le premier contact.
3. Calculer et quantifier le MRCP sur Cz/C3/C4, puis enregistrer les métriques pour un ou plusieurs sujets.


### Qu'est-ce que le paradigme Five-Point?

Le **Five-Point Test** est une tâche de **créativité motrice** où les participants doivent créer des dessins uniques en connectant 5 points disposés de manière fixe. C'est un test classique de **fluence figurale** et de **planification motrice**.

**Déroulement d'un essai:**
1. **Présentation du stimulus** (5 points arrangés) → Trigger "Onset"
2. **Phase d'idéation** (le participant planifie mentalement le dessin)
3. **Premier contact** (début de l'exécution motrice) → Trigger "FirstStroke"
4. **Exécution** (dessin du pattern)
5. **Complétion** ou retouches éventuelles

### Composantes cérébrales étudiées

**1. MRCP (Movement-Related Cortical Potential)**
- **Définition**: Potentiel cortical lent précédant et accompagnant le mouvement volontaire
- **Latence**: Commence ~1-2 secondes AVANT le mouvement
- **Topographie**: Maximum sur les électrodes centrales (Cz, C3, C4) - cortex moteur
- **Phases**:
  - **Bereitschaftspotential (BP)**: Potentiel de préparation (~1.5-0.5s avant)
  - **NS' (Negative Slope')**: Rampe négative tardive (~0.5s-0s avant)
  - **Motor Potential (MP)**: Pic négatif juste avant/pendant le mouvement
- **Interprétation**: Activation progressive du cortex moteur et prémoteur

**2. CNV (Contingent Negative Variation)**
- **Définition**: Onde lente négative entre un stimulus d'avertissement et un stimulus impératif
- **Latence**: Entre le stimulus (onset) et l'action anticipée
- **Topographie**: Maximum frontocentral (Fz, F3, F4, Cz)
- **Composantes**:
  - **CNV précoce**: Orientation de l'attention (~300-600 ms)
  - **CNV tardive**: Préparation motrice (~600 ms jusqu'à l'action)
- **Interprétation**: Anticipation, préparation, et motivation

**3. ERD/ERS (Event-Related Desynchronization/Synchronization)**
- **Définition**: Changements de puissance dans les oscillations cérébrales
- **Bandes de fréquence**:
  - **Mu (8-13 Hz)**: Rythme sensorimoteur
  - **Beta (13-30 Hz)**: Rythme moteur
- **Patterns typiques**:
  - **ERD (désynchronisation)**: DIMINUTION de puissance pendant la préparation/exécution
  - **ERS (synchronisation)**: AUGMENTATION de puissance après le mouvement ("beta rebound")
- **Interprétation**: 
  - ERD = activation du cortex sensorimoteur
  - ERS = inhibition post-mouvement / reset du système moteur



## 0. Préparation et configuration

Nous initialisons l'environnement : imports, chemins BIDS, choix du sujet. Chaque bloc précise son fonctionnement et suggère comment les étudiants peuvent expérimenter (changer de sujet, jouer sur les paramètres de rejet, etc.).


### Bloc Type 1 — Imports et options globales

Charge les bibliothèques principales (MNE, NumPy, Matplotlib, outils BIDS) et fixe quelques réglages d'affichage. Ajoutez ici tout package supplémentaire dont vous auriez besoin (ex. `scipy`).


In [ ]:
# -----------------------------------------------------------------------------
# Imports principaux et configuration globale
# -----------------------------------------------------------------------------
import warnings  # contrôle des avertissements Python (MNE en produit beaucoup)
from pathlib import Path  # manipulation portable des chemins fichiers
import csv  # lecture / écriture de fichiers tabulés (participants.tsv)
from collections import Counter  # comptage des annotations pour diagnostic

import matplotlib.pyplot as plt  # visualisation des signaux et ERP
import mne  # bibliothèque principale pour l'analyse EEG/MEG
import numpy as np  # calcul scientifique vectorisé
from mne_bids import BIDSPath  # construction de chemins compatibles BIDS

warnings.filterwarnings('ignore', category=RuntimeWarning)
mne.set_log_level('INFO')  # niveau d'information raisonnable (DEBUG serait verbeux)
plt.rcParams['figure.figsize'] = (12, 6)  # figure large pour voir plusieurs canaux

print('Versions utilisées:')  # utile pour la reproductibilité entre étudiants
print(' - mne      ', mne.__version__)
print(' - numpy    ', np.__version__)


### Bloc Type 1 — Définir le dossier BIDS et les dérivés

Localise les données Five-Point et prépare les dossiers de sortie :
- `derivatives/preproc` : fichiers prétraités (sortie du notebook 01).
- `derivatives/fivepoint-analysis` : artefacts générés ici (métriques JSON, CSVs).
Vous pouvez créer d'autres sous-dossiers si vous développez de nouvelles analyses.


In [ ]:
# -----------------------------------------------------------------------------
# Chemins BIDS et dérivés (priorité au dossier utilisé dans le notebook 01)
# -----------------------------------------------------------------------------
CANDIDATE_ROOTS = [
    Path('tasks/fivepoint/bids'),             # même chemin que dans 01_preprocessing_notebook.ipynb

]

root_bids = next((path for path in CANDIDATE_ROOTS if path.exists()), None)
if root_bids is None:
    raise FileNotFoundError('Aucun dossier BIDS détecté — ajustez CANDIDATE_ROOTS en fonction de votre machine.')

# Dossiers dérivés harmonisés avec le pipeline de prétraitement
deriv_preproc = root_bids / 'derivatives' / 'preproc'
if not deriv_preproc.exists():
    raise FileNotFoundError('Le dossier preproc est introuvable : lancez le notebook 01 ou vérifiez son emplacement.')

# Dossier pour les artefacts produits par ce carnet (métriques, MRCP sauvegardés)
deriv_analysis = root_bids / 'derivatives' / 'fivepoint-analysis'
deriv_analysis.mkdir(parents=True, exist_ok=True)

TASK_LABEL = 'fivepoint'

print('Chemins vérifiés:')
print(' - BIDS root        :', root_bids.resolve())
print(' - dérivés préproc  :', deriv_preproc.resolve())
print(' - dérivés analyse  :', deriv_analysis.resolve())


### Bloc Type 1 — Lister les participants disponibles

Lit `participants.tsv` pour récupérer les identifiants `sub-XX`. Utilisez cette liste pour choisir le sujet à analyser ou pour préparer une boucle multi-sujets.


In [ ]:
# -----------------------------------------------------------------------------
# Lecture de participants.tsv pour récupérer les identifiants `sub-XX`
# -----------------------------------------------------------------------------
participants_tsv = root_bids / 'participants.tsv'
if not participants_tsv.exists():
    raise FileNotFoundError(f'participants.tsv introuvable: {participants_tsv}')

subjects = []
with participants_tsv.open('r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='	')  # format TSV standard BIDS
    header = next(reader, None)  # ignore l'en-tête si présent
    for row in reader:
        if row and row[0].startswith('sub-'):
            subjects.append(row[0].replace('sub-', ''))

if not subjects:
    raise RuntimeError('Aucun sujet détecté dans participants.tsv')

print('Sujets disponibles :', subjects)


### Bloc Type 2 — Sélectionner un participant et une session

Choisissez le sujet/session/run à analyser. Pour explorer d'autres participants, modifiez ces variables puis relancez les cellules suivantes. Vous trouverez une boucle multi-sujets plus bas pour automatiser cette étape.


In [ ]:
# -----------------------------------------------------------------------------
# Choix du sujet / session / run
# -----------------------------------------------------------------------------
subject = '10'  # <--- changez cette valeur pour explorer un autre participant
session = '001'
run = '01'

print(f'Sujet en cours: sub-{subject}, session {session}, run {run}')


## 1. Charger un enregistrement prétraité

On charge le fichier `*_processed.fif` généré par le pipeline (filtrage + AutoReject + ICA). L’analyse MRCP part d’un signal déjà propre.


### Bloc Type 1 — Fonction utilitaire de chargement

`load_processed_raw` construit un `BIDSPath` vers `derivatives/preproc`, charge le Raw et applique une référence moyenne (utile pour les potentiels lents). Vous pouvez réutiliser cette fonction dans d'autres scripts de motor imagery.


In [ ]:
# -----------------------------------------------------------------------------
# Fonction utilitaire : chargement du fichier prétraité
# -----------------------------------------------------------------------------
def load_processed_raw(subject: str, session: str = '001', run: str = '01'):
    # Charge le fichier `*_processed.fif`, applique la référence moyenne et retourne (Raw, chemin).
    processed_bids = BIDSPath(
        root=deriv_preproc,
        subject=subject,
        session=session,
        task=TASK_LABEL,
        run=run,
        datatype='eeg',
        processing='clean',
        suffix='processed',
        extension='.fif'
    )
    if not processed_bids.fpath.exists():
        raise FileNotFoundError(f"Fichier prétraité introuvable : {processed_bids.fpath}")

    raw_obj = mne.io.read_raw_fif(processed_bids.fpath, preload=True)
    raw_obj.set_eeg_reference('average', projection=False)  # référence moyenne nécessaire pour les potentiels lents
    if raw_obj.get_montage() is None:
        raw_obj.set_montage('standard_1020', match_case=False, on_missing='warn')
    return raw_obj, processed_bids.fpath


### Bloc Type 1 — Charger et inspecter les métadonnées

Affiche des informations clés : fréquence d’échantillonnage, canaux EEG, annotations disponibles. Vérifiez que les événements `Stimulus/S 11/12/13/14/16` sont bien présents (ils servent à remapper les actions dans la tâche).


In [ ]:
# -----------------------------------------------------------------------------
# Chargement du sujet choisi et inspection rapide
# -----------------------------------------------------------------------------
raw, processed_path = load_processed_raw(subject, session=session, run=run)
print('Fichier prétraité chargé :', processed_path)
print("Fréquence d'échantillonnage :", raw.info['sfreq'], "Hz")
print("Canaux EEG                :", len(mne.pick_types(raw.info, eeg=True)))
print("Canaux bad                :", raw.info['bads'])
print("Durée totale (minutes)    :", raw.times[-1] / 60.0)

annotation_counts = Counter(str(desc) for desc in raw.annotations.description)
print("Annotations disponibles   :")
for desc, count in annotation_counts.items():
    print(f' - {desc}: {count}')


### Bloc Type 2 — Visualisation rapide (optionnel)

Décommentez ces lignes pour voir quelques secondes du signal. Ajustez `n_channels`, `scalings`, `tmax` pour illustrer les différents types de segments (repos vs mouvement) aux étudiants.


In [ ]:
# raw.copy().crop(tmax=5).plot(n_channels=12, scalings='auto', title='Signal prétraité (5 s)')


## 2. Préparer les événements et créer les epochs

Nous remapons les annotations vers des codes lisibles (`first_touch`, `retouch`, etc.), puis construisons les epochs autour de l’action principale (premier toucher). Le MRCP est ensuite extrait à partir de ces epochs.


In [ ]:
# -----------------------------------------------------------------------------
# Remappage des annotations vers des labels lisibles
# -----------------------------------------------------------------------------
annotation_map = {
    'Stimulus/S 12': 'onset',
    'Stimulus/S 14': 'onset',  # éventuelle variante
    'Stimulus/S 11': 'first_touch',
    'first_touch': 'first_touch',  # déjà renommé par certaines pipelines
    'Stimulus/S 16': 'retouch',
    'Stimulus/S 13': 'time_limit',
}

events, event_id = mne.events_from_annotations(raw)
selected_events = []
selected_event_id = {}
next_code = 1

for annotation, label in annotation_map.items():
    if annotation not in event_id:
        continue
    if label not in selected_event_id:
        selected_event_id[label] = next_code
        next_code += 1
    new_code = selected_event_id[label]

    subset = events[events[:, 2] == event_id[annotation]].copy()
    if subset.size == 0:
        continue
    subset[:, 2] = new_code
    selected_events.append(subset)

if not selected_events:
    raise RuntimeError('Aucun événement reconnu — vérifiez annotation_map.')

events = np.concatenate(selected_events, axis=0)
events = events[np.argsort(events[:, 0])]  # trie chronologique

counts = {label: int((events[:, 2] == code).sum()) for label, code in selected_event_id.items()}
print('Événements retenus :', selected_event_id)
print('Occurrences       :', counts)


### Bloc Type 2 — Adapter le mapping (optionnel)

Si vous utilisez des annotations personnalisées, complétez `annotation_map_custom` et relancez la cellule précédente. Par exemple, mappez `Stimulus/S 15` vers `retouch` ou vers une classe supplémentaire si vous souhaitez la visualiser.


In [ ]:
# annotation_map_custom = {
#     'Stimulus/S 15': 'retouch',
# }
# annotation_map.update(annotation_map_custom)
# print('annotation_map mis à jour :', annotation_map)


In [ ]:
# -----------------------------------------------------------------------------
# Paramètres MRCP (fenêtre, baseline, canaux)
# -----------------------------------------------------------------------------
mrcp_tmin, mrcp_tmax = -1.5, 0.5
mrcp_baseline = (-1.5, -1.2)
mrcp_channels = ['Cz', 'C3', 'C4']


In [ ]:
# -----------------------------------------------------------------------------
# Construction des epochs pour le MRCP
# -----------------------------------------------------------------------------
if 'first_touch' not in selected_event_id:
    raise RuntimeError('Événement first_touch absent : impossible de calculer le MRCP.')

touch_event_id = {'first_touch': selected_event_id['first_touch']}
touch_events = events[events[:, 2] == touch_event_id['first_touch']]

print(f"Nombre d'essais first_touch: {len(touch_events)}")

epochs_mrcp = mne.Epochs(
    raw,
    touch_events,
    event_id=touch_event_id,
    tmin=mrcp_tmin,
    tmax=mrcp_tmax,
    baseline=mrcp_baseline,
    picks=mrcp_channels,
    preload=True,
    detrend=None,
    reject=None,
    verbose='warning',
)

print(epochs_mrcp)


### Bloc Type 2 — Visualiser quelques epochs (optionnel)

Décommentez ces lignes pour inspecter des single trials. Cherchez des artefacts résiduels ou discutez des patterns inter-essais.


In [ ]:
# epochs_mrcp[:10].plot(picks=mrcp_channels, title='Epochs MRCP (10 premiers essais)')


## 3. Calculer et visualiser le MRCP

Nous moyennons les epochs autour du mouvement pour obtenir le MRCP et sauvegardons le résultat. Des visualisations additionnelles (différence de conditions, topographies) peuvent être ajoutées si vous enrichissez la tâche.


### Bloc Type 1 — Moyennage et tracé du MRCP

Calcule le MRCP moyen, l'affiche et le sauvegarde dans `derivatives/fivepoint-analysis`. Encouragez les étudiants à regarder la latence du minimum ou à superposer plusieurs canaux.


In [ ]:
# -----------------------------------------------------------------------------
# Moyenne Movement-locked et sauvegarde du MRCP
# -----------------------------------------------------------------------------
evoked_mrcp = epochs_mrcp.average()
print(f'MRCP moyenné sur {evoked_mrcp.nave} essais')

evoked_mrcp.plot(spatial_colors=True, time_unit='s', titles='MRCP (movement-locked)')

session_label = session if session is not None else 'NA'
subject_dir = deriv_analysis / f'sub-{subject}'
subject_dir.mkdir(parents=True, exist_ok=True)
evoked_path = subject_dir / f'sub-{subject}_ses-{session_label}_task-{TASK_LABEL}_mrcp-ave.fif'
evoked_mrcp.save(evoked_path, overwrite=True)
print('MRCP sauvegardé :', evoked_path)


### Bloc Type 2 — Calculer des métriques simples (amplitude/latence)

Exemple de quantification sur Cz : amplitude moyenne dans `[-0.8, -0.3] s`, latence du minimum. Invitez les étudiants à tester d'autres canaux/fenêtres et à comparer les résultats entre sujets.


In [ ]:
# -----------------------------------------------------------------------------
# Extraction de métriques simples sur Cz (fenêtre pré-mouvement)
# -----------------------------------------------------------------------------
def mean_amplitude_microvolt(evoked: mne.Evoked, pick: str, tmin: float, tmax: float) -> float:
    evk = evoked.copy().pick(pick)
    start, stop = evk.time_as_index([tmin, tmax])
    segment = evk.data[0, start:stop]
    return float(segment.mean() * 1e6)

def peak_latency(evoked: mne.Evoked, pick: str, tmin: float, tmax: float, mode: str = 'neg') -> tuple[float, float]:
    evk = evoked.copy().pick(pick)
    ch_name, time_s = evk.get_peak(tmin=tmin, tmax=tmax, mode=mode)
    time_idx = evk.time_as_index(time_s)
    amp = evk.data[0, time_idx]
    return float(time_s), float(amp * 1e6)

metrics = {}
channel = 'Cz'
window = (-0.8, -0.3)  # fenêtre où le MRCP s'enfonce
metrics['mean_amp_uV'] = mean_amplitude_microvolt(evoked_mrcp, channel, *window)
metrics['peak_time_s'], metrics['peak_amp_uV'] = peak_latency(evoked_mrcp, channel, *window, mode='neg')

metrics_path = subject_dir / f'sub-{subject}_ses-{session_label}_task-{TASK_LABEL}_mrcp-metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Métriques sauvegardées :', metrics_path)
print(metrics)


## 4. Extension multi-sujets (optionnel)

Boucle sur plusieurs participants pour générer un tableau de synthèse et des MRCP de groupe. Idéal pour un mini-projet où chaque équipe compare les variations de MRCP entre sujets ou conditions.


### Bloc Type 3 — Calculer les métriques pour tous les sujets

Parcours la liste `subjects`, calcule les MRCP individuels et sauvegarde un CSV récapitulatif. Proposez aux étudiants de compléter ce tableau avec d'autres colonnes (ex. latence sur C3/C4, variance du MRCP, etc.).


In [ ]:
# -----------------------------------------------------------------------------
# Boucle multi-sujets : extraction de Cz et résumé CSV
# -----------------------------------------------------------------------------
summary_rows = []
for sub in subjects:
    try:
        raw_sub, _ = load_processed_raw(sub, session=session, run=run)
    except FileNotFoundError:
        print(f'sub-{sub}: fichier prétraité manquant, ignoré.')
        continue

    events_sub, event_id_sub = mne.events_from_annotations(raw_sub)
    selected_sub = []
    for annotation, label in annotation_map.items():
        if annotation not in event_id_sub:
            continue
        sel = events_sub[events_sub[:, 2] == event_id_sub[annotation]].copy()
        if sel.size == 0:
            continue
        code = selected_event_id.get(label, len(selected_event_id) + 1)
        selected_event_id[label] = code
        sel[:, 2] = code
        selected_sub.append(sel)

    if not selected_sub or 'first_touch' not in selected_event_id:
        print(f'sub-{sub}: événements first_touch introuvables, ignoré.')
        continue

    events_sub = np.concatenate(selected_sub, axis=0)
    events_sub = events_sub[np.argsort(events_sub[:, 0])]
    touch_events_sub = events_sub[events_sub[:, 2] == selected_event_id['first_touch']]
    if len(touch_events_sub) == 0:
        print(f'sub-{sub}: aucun essai first_touch, ignoré.')
        continue

    epochs_sub = mne.Epochs(
        raw_sub,
        touch_events_sub,
        event_id={'first_touch': selected_event_id['first_touch']},
        tmin=mrcp_tmin,
        tmax=mrcp_tmax,
        baseline=mrcp_baseline,
        picks=mrcp_channels,
        preload=True,
        detrend=None,
        reject=None,
        verbose='error'
    )
    if len(epochs_sub) == 0:
        print(f'sub-{sub}: epochs vides, ignoré.')
        continue

    evoked_sub = epochs_sub.average()
    subj_metrics = {
        'subject': sub,
        'n_epochs': len(epochs_sub),
        'cz_mean_uV': mean_amplitude_microvolt(evoked_sub, 'Cz', *window),
    }
    summary_rows.append(subj_metrics)

if summary_rows:
    session_label = session if session is not None else 'NA'
    csv_path = deriv_analysis / f'group_mrcp_summary_ses-{session_label}.csv'
    import pandas as pd
    pd.DataFrame(summary_rows).to_csv(csv_path, index=False)
    print('Tableau de synthèse sauvegardé :', csv_path)
else:
    print('Aucun sujet qualifié pour la synthèse multi-sujets.')
